# Reproducibility smoke test on the free sample

This notebook runs `../validation/benchmark_script.py`, **unmodified and identical to the script used for the published benchmark**, against the free sample shipped in `../sample/`.

**Purpose:** show that the code runs end to end against the real delivered schema.

**Not the purpose:** reproduce the published metric. The sample holds 500 of the 20,000 subjects, and the benchmark splits it into a 400/100 train/test split plus 5-fold cross-validation inside the training part. A different random draw of 500 subjects would move every figure below.

The number to cite is the one measured on the full dataset: held-out test AUC 0.719, F1 0.721, Brier 0.207, calibration slope 0.88 (n = 20,000, task: GOSE < 8 at 12 months).


In [1]:
import sys, subprocess, shutil, tempfile, pathlib
import pandas as pd
HERE = pathlib.Path.cwd()
SAMPLE = (HERE / ".." / "sample").resolve()
SCRIPT = (HERE / ".." / "validation" / "benchmark_script.py").resolve()
work = pathlib.Path(tempfile.mkdtemp(prefix="smoke_"))   # all outputs go here, nothing is written into the repository
print("sample dir :", SAMPLE.name)
print("script     :", SCRIPT.name, "(unmodified copy of the published benchmark script)")
def clean(text):
    return text.replace(str(work), "<tmp>").replace(str(SAMPLE), "<sample>")


sample dir : sample
script     : benchmark_script.py (unmodified copy of the published benchmark script)


In [2]:
shutil.copy(SCRIPT, work / "benchmark_script.py")
shutil.copy(SAMPLE / "btbi_sample_500.parquet", work / "btbi_sample_500.parquet")
r = subprocess.run([sys.executable, "benchmark_script.py", "btbi_sample_500.parquet"],
                   cwd=work, capture_output=True, text=True)
print(clean(r.stdout))
print(clean(r.stderr[-1500:]))
r.check_returncode()
results = pd.read_parquet(work / "benchmark_results.parquet")
results

dataset: btbi_sample_500.parquet  shape=(500, 78)
label prevalence (GOSE<8 at 12 mo): 0.5880
fold 1: AUC=0.6512 F1=0.6531 brier=0.2343
fold 2: AUC=0.6505 F1=0.7234 brier=0.2369
fold 3: AUC=0.6480 F1=0.6875 brier=0.2393
fold 4: AUC=0.6460 F1=0.7010 brier=0.2396
fold 5: AUC=0.6802 F1=0.7273 brier=0.2220
train: AUC=0.7818 | test: AUC=0.6912 F1=0.6885 brier=0.2263 cal_slope=0.5631
saved -> <tmp>/benchmark_results.parquet (7, 9)




,split,n_train,n_eval,prevalence,auc,f1,brier,calibration_slope,calibration_intercept
0,cv_fold_1,320,80,0.5875,0.651193,0.653061,0.234253,0.573675,0.062663
1,cv_fold_2,320,80,0.5875,0.650548,0.723404,0.236942,0.500110,0.269960
2,cv_fold_3,320,80,0.5875,0.647969,0.687500,0.239321,0.324476,0.207822
3,cv_fold_4,320,80,0.5875,0.646035,0.701031,0.239585,0.356786,0.162533
4,cv_fold_5,320,80,0.5875,0.680206,0.727273,0.222016,0.608088,0.064562
5,train_full,400,400,0.5875,0.781844,0.767010,0.187184,1.134993,-0.029772
6,test,400,100,0.5900,0.691195,0.688525,0.226278,0.563094,0.127959


## Reading the output

The script prints per-fold AUC/F1/Brier for 5-fold cross-validation on the training part and then the held-out test line. On 500 rows (about 100 test subjects) these figures are noisy and sit below the published full-dataset value; that is sample size, not a defect in the script. In the executed run the cross-validation folds give AUC 0.646 to 0.680, and the held-out test set (100 subjects) gives AUC 0.691, F1 0.689 and calibration slope 0.56, against the published 0.719, 0.721 and 0.88.